In [ ]:
# ==========================================================
# Cell 1 - Install Required Libraries
# ==========================================================

!pip -q install -U \
langchain \
langchain-openai \
langchain-core \
pydantic \
python-dotenv \
tabulate

In [ ]:
# ==========================================================
# Cell 2 - Import Required Libraries
# ==========================================================

import os
import json
import sqlite3
import hashlib
import requests

from typing import Any, Dict, List, Literal

from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

from langchain_core.runnables import RunnableLambda

from langchain_openai import ChatOpenAI

print("✅ All libraries imported successfully.")

In [ ]:
# ==========================================================
# Cell 3 - Configuration
# ==========================================================

# -------------------------------
# Database
# -------------------------------
DB_NAME = "meeting_actions.db"

# -------------------------------
# Safety
# -------------------------------
MAX_TOOL_STEPS = 5

# -------------------------------
# GitHub API
# -------------------------------
GITHUB_API_URL = "https://api.github.com"

# -------------------------------
# LLM
# -------------------------------
MODEL_NAME = "openai/gpt-4o-mini"

# -------------------------------
# Email Mode
# -------------------------------
# "mock" -> no email will be sent
# "live" -> sends real emails
EMAIL_MODE = "mock"

print("Configuration Loaded Successfully")

In [ ]:
# ==========================================================
# Cell 4 - Load Secrets from Google Colab
# ==========================================================

from google.colab import userdata

# -------------------------------
# OpenRouter
# -------------------------------
OPEN_ROUTER_API_KEY = userdata.get("OPEN_ROUTER_API_KEY")

# -------------------------------
# GitHub
# -------------------------------
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GITHUB_OWNER = userdata.get("GITHUB_OWNER")
GITHUB_REPO = userdata.get("GITHUB_REPO")

# -------------------------------
# Validate
# -------------------------------
required = {
    "OPEN_ROUTER_API_KEY": OPEN_ROUTER_API_KEY,
    "GITHUB_TOKEN": GITHUB_TOKEN,
    "GITHUB_OWNER": GITHUB_OWNER,
    "GITHUB_REPO": GITHUB_REPO,
}

missing = [key for key, value in required.items() if not value]

if missing:
    raise ValueError(
        f"Missing Colab Secret(s): {', '.join(missing)}"
    )

print("✅ All secrets loaded successfully.")

In [ ]:
# ==========================================================
# Cell 5 - SQLite Database Initialization
# ==========================================================

def initialize_database():
    """
    Creates the SQLite database and action_items table
    if they do not already exist.
    """

    connection = sqlite3.connect(DB_NAME)
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS action_items (

            id INTEGER PRIMARY KEY AUTOINCREMENT,

            request_id TEXT NOT NULL,

            action_title TEXT NOT NULL,

            owner TEXT,

            commitment TEXT NOT NULL,

            status TEXT NOT NULL,

            reason TEXT,

            github_issue_number INTEGER,

            github_issue_url TEXT,

            transcript_evidence TEXT,

            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

            UNIQUE(request_id, action_title)
        )
    """)

    connection.commit()
    connection.close()

    print("✅ Database initialized successfully.")


# Create the database
initialize_database()

In [ ]:
# ==========================================================
# Cell 6 - Database Helper Functions
# ==========================================================

def save_action_item(record: dict):
    """
    Insert or update an action item in the SQLite database.
    """

    connection = sqlite3.connect(DB_NAME)
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO action_items (
            request_id,
            action_title,
            owner,
            commitment,
            status,
            reason,
            github_issue_number,
            github_issue_url,
            transcript_evidence
        )

        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)

        ON CONFLICT(request_id, action_title)
        DO UPDATE SET

            owner = excluded.owner,
            commitment = excluded.commitment,
            status = excluded.status,
            reason = excluded.reason,
            github_issue_number = excluded.github_issue_number,
            github_issue_url = excluded.github_issue_url,
            transcript_evidence = excluded.transcript_evidence

        """,
        (
            record.get("request_id"),
            record.get("title"),
            record.get("owner"),
            record.get("commitment"),
            record.get("status"),
            record.get("reason"),
            record.get("github_issue_number"),
            record.get("github_issue_url"),
            record.get("evidence"),
        ),
    )

    connection.commit()
    connection.close()


def get_existing_action(request_id: str, title: str):
    """
    Returns an existing action item if already stored.
    """

    connection = sqlite3.connect(DB_NAME)
    connection.row_factory = sqlite3.Row

    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT *
        FROM action_items
        WHERE request_id = ?
        AND action_title = ?
        """,
        (request_id, title),
    )

    row = cursor.fetchone()

    connection.close()

    if row:
        return dict(row)

    return None


def get_all_actions():
    """
    Returns every action stored in the database.
    """

    connection = sqlite3.connect(DB_NAME)
    connection.row_factory = sqlite3.Row

    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT *
        FROM action_items
        ORDER BY created_at DESC
        """
    )

    rows = cursor.fetchall()

    connection.close()

    return [dict(row) for row in rows]


print("✅ Database helper functions are ready.")

In [ ]:
# ==========================================================
# Cell 7 - Pydantic Schema for GitHub Tool
# ==========================================================

class GitHubIssueInput(BaseModel):
    """
    Input schema used by the LangChain GitHub tool.
    """

    request_id: str = Field(
        description="Unique request identifier"
    )

    title: str = Field(
        min_length=5,
        description="Short GitHub issue title"
    )

    description: str = Field(
        min_length=10,
        description="Detailed task description"
    )

    owner: str | None = Field(
        default=None,
        description="Person responsible for the task"
    )

    commitment: Literal[
        "confirmed",
        "tentative",
        "suggestion",
    ]

    evidence: str = Field(
        description="Supporting statement from the meeting transcript"
    )


print("✅ GitHubIssueInput schema created successfully.")

In [ ]:
# ==========================================================
# Cell 8 - GitHub Helper Functions
# ==========================================================

def create_issue_on_github(title: str, body: str) -> dict:
    """
    Create a GitHub issue and return a normalized response.
    """

    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/issues"

    try:
        response = requests.post(
            url,
            headers=headers,
            json={
                "title": title,
                "body": body,
            },
            timeout=20,
        )

    except requests.RequestException as error:

        return {
            "status": "failed",
            "reason": str(error),
        }

    if response.status_code == 201:

        issue = response.json()

        return {
            "status": "created",
            "issue_number": issue["number"],
            "issue_url": issue["html_url"],
        }

    if response.status_code == 422:

        message = response.json().get("message", "")

        if "already exists" in message.lower():

            return {
                "status": "already_exists",
                "reason": message,
            }

    return {
        "status": "failed",
        "status_code": response.status_code,
        "reason": response.json().get(
            "message",
            "Unknown GitHub error",
        ),
    }


print("✅ GitHub helper function created successfully.")

In [ ]:
# ==========================================================
# Cell 9 - LangChain Tool
# ==========================================================

@tool(args_schema=GitHubIssueInput)
def create_github_issue(
    request_id: str,
    title: str,
    description: str,
    commitment: str,
    evidence: str,
    owner: str | None = None,
) -> dict:
    """
    Create a GitHub issue only for confirmed action items.
    """

    # Check whether this action already exists locally
    existing = get_existing_action(request_id, title)

    if existing and existing.get("status") == "created":

        return {
            "request_id": request_id,
            "title": title,
            "status": "already_exists",
            "reason": "Action already stored for this request_id",
            "issue_number": existing.get("github_issue_number"),
            "issue_url": existing.get("github_issue_url"),
        }

    # Only confirmed actions become GitHub issues
    if commitment != "confirmed":

        result = {
            "request_id": request_id,
            "title": title,
            "owner": owner,
            "commitment": commitment,
            "evidence": evidence,
            "status": "skipped",
            "reason": f"Action is {commitment}, not confirmed",
        }

        save_action_item(result)

        return result

    issue_body = f"""
## Description

{description}

## Owner

{owner or "Unassigned"}

## Meeting Evidence

> {evidence}

## Request ID

`{request_id}`
""".strip()

    github_result = create_issue_on_github(
        title=title,
        body=issue_body,
    )

    result = {
        "request_id": request_id,
        "title": title,
        "owner": owner,
        "commitment": commitment,
        "evidence": evidence,
        **github_result,
    }

    save_action_item(result)

    return result


print("✅ LangChain tool created successfully.")

In [ ]:
# ==========================================================
# Cell 9 - Action Key Generator
# ==========================================================

def action_key(request_id: str, title: str) -> str:
    """
    Create a stable identifier for an action item.
    Used for idempotency and duplicate detection.
    """

    digest = hashlib.sha256(
        f"{request_id}::{title.strip().lower()}".encode("utf-8")
    ).hexdigest()[:12]

    return f"{request_id}:{digest}"


print("✅ Action key generator ready.")

In [ ]:
# ==========================================================
# Cell 10 - LangChain Tool
# ==========================================================

@tool(args_schema=GitHubIssueInput)
def create_github_issue(
    request_id: str,
    title: str,
    description: str,
    commitment: str,
    evidence: str,
    owner: str | None = None,
) -> dict:
    """
    Create a GitHub issue only for confirmed action items.
    """

    # Check whether this action already exists locally
    existing = get_existing_action(request_id, title)

    if existing and existing.get("status") == "created":

        return {
            "request_id": request_id,
            "title": title,
            "status": "already_exists",
            "reason": "Action already stored for this request_id",
            "issue_number": existing.get("github_issue_number"),
            "issue_url": existing.get("github_issue_url"),
        }

    # Only confirmed actions become GitHub issues
    if commitment != "confirmed":

        result = {
            "request_id": request_id,
            "title": title,
            "owner": owner,
            "commitment": commitment,
            "evidence": evidence,
            "status": "skipped",
            "reason": f"Action is {commitment}, not confirmed",
        }

        save_action_item(result)

        return result

    issue_body = f"""
## Description

{description}

## Owner

{owner or "Unassigned"}

## Meeting Evidence

> {evidence}

## Request ID

`{request_id}`
""".strip()

    github_result = create_issue_on_github(
        title=title,
        body=issue_body,
    )

    result = {
        "request_id": request_id,
        "title": title,
        "owner": owner,
        "commitment": commitment,
        "evidence": evidence,
        **github_result,
    }

    save_action_item(result)

    return result


print("✅ LangChain tool created successfully.")

In [ ]:
# ==========================================================
# Cell 11 - LLM & Tool Binding
# ==========================================================

tools = [create_github_issue]

tools_repo = {
    tool.name: tool
    for tool in tools
}

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPEN_ROUTER_API_KEY,
    model="openai/gpt-4o-mini",
    temperature=0,
    max_tokens=1500,
)

tool_llm = llm.bind_tools(tools)

print("✅ LLM initialized and tools bound successfully.")

In [ ]:
# ==========================================================
# Cell 12 - Prompt Template
# ==========================================================

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You extract action items from meeting transcripts.

Classify every actionable statement as:

- confirmed: a clear commitment
- tentative: conditional, unclear, or unresolved work
- suggestion: an idea without a firm commitment

Use the create_github_issue tool once for every actionable item.

Never invent or modify request_id.

Use transcript evidence exactly as support for each classification.
""".strip(),
        ),
        (
            "human",
            """
Request ID: {request_id}

Meeting Transcript:

{meeting_transcript}
""".strip(),
        ),
    ]
)

print("✅ Prompt template created successfully.")

In [ ]:
# ==========================================================
# Cell 13 - Call Tool LLM
# ==========================================================

def call_tool_llm(prompt_value):
    """
    Convert the prompt into LangChain messages and invoke
    the tool-enabled LLM once.
    """

    messages = prompt_value.to_messages()

    ai_message = tool_llm.invoke(messages)

    return {
        "messages": messages,
        "ai_message": ai_message,
    }


print("✅ Tool LLM caller created successfully.")

In [ ]:
# ==========================================================
# Cell 14 - Manual Tool Loop
# ==========================================================

def run_tool_loop(state):
    """
    Execute the LangChain tool loop until the model
    stops requesting tool calls.
    """

    messages = state["messages"]
    ai_message = state["ai_message"]

    outcomes = []

    for _ in range(MAX_TOOL_STEPS):

        messages.append(ai_message)

        if not ai_message.tool_calls:

            return {
                "status": "completed",
                "steps": len(outcomes) + 1,
                "outcomes": outcomes,
                "summary": ai_message.content,
            }

        for tool_call in ai_message.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]

            if tool_name not in tools_repo:

                tool_result = {
                    "request_id": tool_args.get("request_id"),
                    "title": tool_args.get("title"),
                    "status": "failed",
                    "reason": f"Unknown tool: {tool_name}",
                }

            else:

                try:
                    tool_result = tools_repo[tool_name].invoke(tool_args)

                except Exception as error:

                    tool_result = {
                        "request_id": tool_args.get("request_id"),
                        "title": tool_args.get("title"),
                        "status": "failed",
                        "reason": str(error),
                    }

            outcomes.append(tool_result)

            messages.append(
                ToolMessage(
                    content=json.dumps(tool_result, default=str),
                    tool_call_id=tool_call_id,
                )
            )

        ai_message = tool_llm.invoke(messages)

    return {
        "status": "failed",
        "reason": f"Maximum tool-loop steps exceeded: {MAX_TOOL_STEPS}",
        "outcomes": outcomes,
    }


print("✅ Manual tool loop created successfully.")

In [ ]:
# ==========================================================
# Cell 15 - Build Chain & Run Workflow
# ==========================================================

chain = (
    prompt
    | RunnableLambda(call_tool_llm)
    | RunnableLambda(run_tool_loop)
)


def run_meeting_transcript(
    request_id: str,
    meeting_transcript: str,
):
    """
    Run the complete workflow:
    Transcript -> LLM -> Tool Calls -> GitHub -> SQLite
    """

    result = chain.invoke(
        {
            "request_id": request_id,
            "meeting_transcript": meeting_transcript,
        }
    )

    result["actions"] = get_all_actions()

    return result


print("✅ Workflow chain created successfully.")

In [ ]:
# ==========================================================
# Cell 16 - End-to-End Test
# ==========================================================

meeting_transcript = """
Project Kickoff Meeting

John: We need to finish the authentication module by Friday.
Sarah: I'll update the API documentation tomorrow.
Mike: We should consider adding dark mode in a future release.
Emily: I'll investigate the production bug affecting login.
David: Maybe we can migrate to PostgreSQL next quarter.
"""

result = run_meeting_transcript(
    request_id="REQ-001",
    meeting_transcript=meeting_transcript,
)

print(json.dumps(result, indent=2, default=str))